# Thu thập giá OHLCV (`ingest_price_data`) — Luồng 1

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.

**Mục đích:** Tải lịch sử giá OHLCV có cấu trúc qua thư viện **vnstock** , chuẩn hóa/làm sạch, ghi vào **Data Lake** dạng Parquet theo partition Hive-style `date=YYYY-MM-DD` (ngày chạy pipeline).

**Nguồn `Quote`:** ưu tiên **KBS** (`PRIMARY_QUOTE_SOURCE`), fallback **VCI** (`FALLBACK_QUOTE_SOURCE`).

**Đầu ra:** `../data-lake/raw/price/date=<YYYY-MM-DD>/<Mã>.parquet` (một file mỗi mã).

**Cách chạy:** Chọn kernel Python của project (venv), chạy lần lượt các ô: import → helper → định nghĩa hàm → `main()` → ô kiểm tra dữ liệu cuối cùng.


In [1]:
# vnstock Quote: PRIMARY = KBS, FALLBACK = VCI (nguồn ổn định).
# TCBS đã deprecated theo tài liệu vnstock — không dùng làm source.
import logging
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
from vnstock import Listing, Company, Finance, Quote

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

TICKERS: list[str] = [
    # VN30
    "ACB",
    "BCM",
    "BID",
    "BVH",
    "CTG",
    "FPT",
    "GAS",
    "GVR",
    "HDB",
    "HPG",
    "MBB",
    "MSN",
    "MWG",
    "PLX",
    "POW",
    "SAB",
    "SHB",
    "SSI",
    "STB",
    "TCB",
    "TPB",
    "VCB",
    "VHM",
    "VIB",
    "VIC",
    "VJC",
    "VNM",
    "VPB",
    "VRE",
    "VGC",

    
    "DGC",
    "DPM",
    "DCM",
    "HSG",
    "NKG",
    "KDC",
    "QNS",
    "SCS",
    "ACV",
    "VTP",
    "KBC",
    "NVL",
    "PDR",
    "DXG",
    "KDH",
    "AGG",
    "HDG",
    "REE",
    "PNJ",
    "GMD",
]
PRIMARY_QUOTE_SOURCE = "kbs"
FALLBACK_QUOTE_SOURCE = "vci"

INDEX_TICKERS: list[str] = ["VNINDEX", "VN30", "HNXINDEX", "UPCOMINDEX"]

assert len(TICKERS) == 50
assert len(set(TICKERS)) == len(TICKERS)

In [2]:
from vnstock import Quote
import pandas as pd

q = Quote(source="kbs", symbol="VNM")
df_raw = q.history(start="2024-01-01", end="2024-12-31", interval="1D")

print("=== TẤT CẢ CỘT VNSTOCK TRẢ VỀ ===")
print(df_raw.columns.tolist())
print()
print("=== KIỂU DỮ LIỆU ===")
print(df_raw.dtypes)
print()
print("=== 3 DÒNG ĐẦU ===")
print(df_raw.head(3).to_string())
print()
print("=== THỐNG KÊ ===")
print(df_raw.describe())


Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1597, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 25: character maps to <undefined>
Exception in thread Thread-15 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Programs\Pyth

=== TẤT CẢ CỘT VNSTOCK TRẢ VỀ ===
['time', 'open', 'high', 'low', 'close', 'volume']

=== KIỂU DỮ LIỆU ===
time      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object

=== 3 DÒNG ĐẦU ===
                 time   open   high    low  close   volume
0 2024-01-02 07:00:00  59.65  59.91  59.39  59.74  2142800
1 2024-01-03 07:00:00  59.83  60.35  59.48  60.35  1889800
2 2024-01-04 07:00:00  60.44  60.79  60.35  60.35  3401500

=== THỐNG KÊ ===
                             time        open        high        low  \
count                         250  250.000000  250.000000  250.00000   
mean   2024-07-05 06:25:26.400000   60.654200   61.085360   60.21136   
min           2024-01-02 07:00:00   56.700000   56.970000   55.82000   
25%           2024-04-04 13:00:00   58.822500   59.187500   58.54000   
50%           2024-07-06 19:00:00   59.630000   59.990000   59.18000   
75%           2024-10-0

In [3]:
def save_raw(df: pd.DataFrame, category: str, run_date: str, name: str) -> str:
    """Lưu raw dạng song song Parquet + CSV (encoding utf-8-sig).

    Path:
      ../data-lake/raw/{category}/date={run_date}/{name}.parquet
      ../data-lake/raw/{category}/date={run_date}/{name}.csv
    """
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data-lake", "raw", category))
    partition = os.path.join(base_dir, f"date={run_date}")
    os.makedirs(partition, exist_ok=True)

    # Đối với ticker, luôn chuẩn hóa UPPER để đồng nhất tên file.
    if category in ("price", "index", "foreign_trading", "financial_ratio"):
        safe_name = str(name).upper().strip()
    else:
        safe_name = str(name).strip()

    parquet_path = os.path.join(partition, f"{safe_name}.parquet")
    csv_path = os.path.join(partition, f"{safe_name}.csv")

    try:
        df.to_parquet(parquet_path, engine="pyarrow", index=False)
    except ImportError:
        df.to_parquet(parquet_path, engine="fastparquet", index=False)

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    logger.info("Đã ghi %s dòng → %s", len(df), parquet_path)
    logger.info("Đã xuất %s dòng → %s", len(df), csv_path)
    return parquet_path


In [4]:
def _ensure_ingestion_workdir() -> None:
    """Đặt cwd là thư mục chứa notebook để ../data-lake đúng partition."""
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_price_data.ipynb").is_file():
        target = cwd
    elif (cwd / "ingestion" / "ingest_price_data.ipynb").is_file():
        target = cwd / "ingestion"
    else:
        target = cwd
    os.chdir(str(target))


def _date_years_ago(d: date, years: int) -> date:
    try:
        return d.replace(year=d.year - years)
    except ValueError:
        return d.replace(year=d.year - years, day=28)

In [5]:
def fetch_ticker_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Tải OHLCV qua vnstock Quote: thử PRIMARY_QUOTE_SOURCE (KBS), rồi FALLBACK_QUOTE_SOURCE (VCI).
    """
    sym = ticker.upper().strip()
    sources: tuple[str, ...] = (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE)
    last_err: Exception | None = None

    for src in sources:
        try:
            quote = Quote(source=src, symbol=sym)
            df = quote.history(start=start, end=end, interval="1D")
            if df is not None and not df.empty:
                logger.info("Đã lấy dữ liệu %s từ nguồn %s", sym, src.upper())
                return df.copy()
        except Exception as e:
            last_err = e
            logger.debug("Quote source=%s symbol=%s thất bại: %s", src, sym, e)
            continue

    if last_err is not None:
        logger.warning(
            "Không lấy được dữ liệu %s sau khi thử các nguồn: %s",
            sym,
            last_err,
        )
    return pd.DataFrame()


def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa cột, time, lọc volume, quy đổi giá nghìn đồng nếu cần, cờ bất thường."""
    if df.empty:
        return df.copy()

    out = df.copy()
    rename_map: dict[str, str] = {}
    for col in out.columns:
        key = str(col).strip().lower().replace(" ", "")
        if key in ("tradingdate", "trading_date", "date", "ngay"):
            rename_map[col] = "time"
        else:
            rename_map[col] = key
    out = out.rename(columns=rename_map)

    required = {"time", "open", "high", "low", "close", "volume"}
    missing = required - set(out.columns)
    if missing:
        raise KeyError(f"Thiếu cột sau khi chuẩn hóa: {missing}")

    out = out.loc[:, list(required)]
    out["time"] = pd.to_datetime(out["time"], errors="coerce")
    out = out.dropna(subset=["time"])

    out["_vol_num"] = pd.to_numeric(out["volume"], errors="coerce")
    out = out.loc[(out["_vol_num"].notna()) & (out["_vol_num"] > 0)].copy()
    out["volume"] = out["_vol_num"].astype("int64")
    out = out.drop(columns=["_vol_num"])

    for col in ("open", "high", "low", "close"):
        out[col] = pd.to_numeric(out[col], errors="coerce")

    ohlc_max = out[["open", "high", "low", "close"]].max(axis=1, numeric_only=True).max()
    if pd.notna(ohlc_max) and float(ohlc_max) > 10_000:
        for col in ("open", "high", "low", "close"):
            out[col] = out[col] / 1000.0

    out["is_suspicious"] = (out["close"] < 0) | (out["high"] < out["low"])
    out = out.sort_values("time", ascending=True).reset_index(drop=True)
    return out


PRICE_TARGET_COLS: list[str] = [
    "ticker",
    "tradingDate",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "value",
    "source",
    "ingested_at",
]


def build_price_target_schema(
    df: pd.DataFrame,
    ticker: str,
    run_date: str,
    source: str = PRIMARY_QUOTE_SOURCE,
) -> pd.DataFrame:
    """Đảm bảo output price luôn có đủ schema tối thiểu."""
    out = df.copy()

    # Đồng nhất tên cột thời gian về tradingDate nếu transform_data còn trả time.
    if "tradingDate" not in out.columns and "time" in out.columns:
        out = out.rename(columns={"time": "tradingDate"})

    out["ticker"] = ticker
    out["source"] = source
    out["ingested_at"] = pd.to_datetime(run_date)

    for col in PRICE_TARGET_COLS:
        if col not in out.columns:
            out[col] = pd.NA

    extra_cols = [c for c in out.columns if c not in PRICE_TARGET_COLS]
    return out[PRICE_TARGET_COLS + extra_cols]


def save_to_datalake(df: pd.DataFrame, ticker: str, run_date: str) -> str:
    """Giữ hàm cũ để tương thích; thực tế gọi save_raw(category='price')."""
    return save_raw(df, "price", run_date, ticker)

In [6]:
def main() -> None:
    _ensure_ingestion_workdir()
    run_dt = date.today()
    run_date = run_dt.isoformat()
    start_dt = _date_years_ago(run_dt, 3)
    start_s = start_dt.isoformat()
    end_s = run_dt.isoformat()

    logger.info(
        "Partition date=%s | cửa sổ vnstock history %s → %s",
        run_date,
        start_s,
        end_s,
    )

    total = len(TICKERS)
    for idx, ticker in enumerate(TICKERS):
        logger.info("[%s/%s] Đang tải %s...", idx + 1, total, ticker)
        try:
            raw = fetch_ticker_data(ticker, start_s, end_s)
            if raw.empty:
                logger.warning("Bỏ qua %s — không có dữ liệu thô", ticker)
            else:
                cleaned = transform_data(raw)
                if cleaned.empty:
                    logger.warning("Bỏ qua %s — sau làm sạch không còn dòng", ticker)
                else:
                    final_df = build_price_target_schema(cleaned, ticker, run_date)
                    save_raw(final_df, "price", run_date, ticker)
                    logger.info(
                        "Thành công %s (%s dòng)",
                        ticker,
                        len(final_df),
                    )
        except Exception:
            logger.exception("Lỗi khi xử lý mã %s — tiếp tục mã kế tiếp", ticker)

        if idx < total - 1:
            time.sleep(4)

    # Thu thập chỉ số thị trường (index)
    total_idx = len(INDEX_TICKERS)
    for idx, ticker in enumerate(INDEX_TICKERS):
        logger.info("[%s/%s] Đang tải chỉ số %s...", idx + 1, total_idx, ticker)
        try:
            raw = fetch_ticker_data(ticker, start_s, end_s)
            if raw.empty:
                logger.warning("Bỏ qua chỉ số %s — không có dữ liệu thô", ticker)
            else:
                save_raw(raw, "index", run_date, ticker)
                logger.info("Thành công chỉ số %s (%s dòng)", ticker, len(raw))
        except Exception:
            logger.exception("Lỗi khi xử lý chỉ số %s — tiếp tục", ticker)

        if idx < total_idx - 1:
            time.sleep(2)

    logger.info(
        "Hoàn tất pipeline cho %s mã (price) + %s chỉ số (index).",
        total,
        total_idx,
    )

main()

2026-04-03 17:10:38 [INFO] Partition date=2026-04-03 | cửa sổ vnstock history 2023-04-03 → 2026-04-03
2026-04-03 17:10:38 [INFO] [1/50] Đang tải ACB...
2026-04-03 17:10:38 [INFO] Đã lấy dữ liệu ACB từ nguồn KBS
2026-04-03 17:10:38 [INFO] Đã ghi 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\ACB.parquet
2026-04-03 17:10:38 [INFO] Đã xuất 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\ACB.csv
2026-04-03 17:10:38 [INFO] Thành công ACB (749 dòng)
2026-04-03 17:10:42 [INFO] [2/50] Đang tải BCM...
2026-04-03 17:10:43 [INFO] Đã lấy dữ liệu BCM từ nguồn KBS
2026-04-03 17:10:43 [INFO] Đã ghi 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\BCM.parquet
2026-04-03 17:10:43 [INFO] Đã xuất 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\BCM.csv
2026-04-03 17:10:43 [INFO] Thành công BCM (749 dòng)
2026-04-03 17:10:47 

In [7]:
def fetch_foreign_trading(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Lấy khối ngoại theo ngày nếu vnstock trả về các cột foreign_*.

    Không raise exception; nếu thiếu cột hoặc lỗi sẽ trả về DataFrame rỗng.
    """
    import re

    sym = ticker.upper().strip()
    required = [
        "foreign_buy_vol",
        "foreign_sell_vol",
        "foreign_net_vol",
        "foreign_buy_value",
        "foreign_sell_value",
    ]

    def _norm_col(s: str) -> str:
        return re.sub(r"[\s_]+", "", str(s).strip().lower())

    try:
        quote = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        df = pd.DataFrame()
        try:
            # Một số version vnstock hỗ trợ thêm tham số để trả về khối ngoại.
            df = quote.history(start=start, end=end, interval="1D", foreign=True)
        except TypeError:
            # Nếu không hỗ trợ tham số, fallback gọi history cơ bản.
            df = quote.history(start=start, end=end, interval="1D")

        if df is None or df.empty:
            return pd.DataFrame()

        col_map = {_norm_col(c): c for c in df.columns}
        required_norm = [_norm_col(c) for c in required]
        missing_norm = [rn for rn in required_norm if rn not in col_map]
        if missing_norm:
            logger.warning(
                "Không thấy đủ cột foreign_* cho %s (thiếu: %s)",
                sym,
                missing_norm,
            )
            return pd.DataFrame()

        # Giữ cột time nếu có
        time_candidates = ["time", "tradingdate", "trading_date", "date", "ngay"]
        time_col = None
        for cand in time_candidates:
            nc = _norm_col(cand)
            if nc in col_map:
                time_col = col_map[nc]
                break

        keep_cols = ([] if time_col is None else [time_col]) + [col_map[rn] for rn in required_norm]
        out = df.loc[:, keep_cols].copy()
        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_foreign_trading lỗi %s: %s", sym, e)
        return pd.DataFrame()


# --- Run foreign trading ---
_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()
start_dt = _date_years_ago(run_dt, 3)
start_s = start_dt.isoformat()
end_s = run_dt.isoformat()

logger.info("Bắt đầu thu thập foreign_trading | %s → %s", start_s, end_s)

total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải foreign trading %s...", idx + 1, total, ticker)
    try:
        df_foreign = fetch_foreign_trading(ticker, start_s, end_s)
        if df_foreign.empty:
            logger.warning("Bỏ qua %s — không có dữ liệu foreign", ticker)
        else:
            save_raw(df_foreign, "foreign_trading", run_date, ticker)
            logger.info("Thành công foreign %s (%s dòng)", ticker, len(df_foreign))
    except Exception:
        logger.exception("Lỗi khi xử lý foreign %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(4)

logger.info("Hoàn tất foreign_trading cho %s mã.", total)


2026-04-03 17:14:21 [INFO] Bắt đầu thu thập foreign_trading | 2023-04-03 → 2026-04-03
2026-04-03 17:14:21 [INFO] [1/50] Đang tải foreign trading ACB...
2026-04-03 17:14:22 [WARNING] Không thấy đủ cột foreign_* cho ACB (thiếu: ['foreignbuyvol', 'foreignsellvol', 'foreignnetvol', 'foreignbuyvalue', 'foreignsellvalue'])
2026-04-03 17:14:22 [WARNING] Bỏ qua ACB — không có dữ liệu foreign
2026-04-03 17:14:26 [INFO] [2/50] Đang tải foreign trading BCM...
2026-04-03 17:14:26 [WARNING] Không thấy đủ cột foreign_* cho BCM (thiếu: ['foreignbuyvol', 'foreignsellvol', 'foreignnetvol', 'foreignbuyvalue', 'foreignsellvalue'])
2026-04-03 17:14:26 [WARNING] Bỏ qua BCM — không có dữ liệu foreign
2026-04-03 17:14:30 [INFO] [3/50] Đang tải foreign trading BID...
2026-04-03 17:14:31 [WARNING] Không thấy đủ cột foreign_* cho BID (thiếu: ['foreignbuyvol', 'foreignsellvol', 'foreignnetvol', 'foreignbuyvalue', 'foreignsellvalue'])
2026-04-03 17:14:31 [WARNING] Bỏ qua BID — không có dữ liệu foreign
2026-04-03 

In [8]:
def fetch_listing() -> pd.DataFrame:
    """Lấy danh sách mã niêm yết (master data)."""

    def _standardize_columns(df_in: pd.DataFrame) -> pd.DataFrame:
        out = df_in.copy()
        out.columns = [str(c).strip().lower() for c in out.columns]
        return out

    keep_cols = [
        "ticker",
        "organ_name",
        "en_organ_name",
        "exchange",
        "industry_code",
        "industry_name",
        "status",
        "first_listed_date",
    ]

    sources: tuple[str, ...] = (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE)
    last_err: Exception | None = None

    for src in sources:
        try:
            listing = Listing(source=src)
            df = listing.all_symbols()
            if df is None or (hasattr(df, "empty") and df.empty):
                logger.warning("all_symbols() trả rỗng với Listing source=%s", src)
                continue

            df = _standardize_columns(df)
            present = [c for c in keep_cols if c in df.columns]
            df_keep = df.loc[:, present].copy() if present else df.copy()

            if "exchange" in df_keep.columns:
                df_keep["exchange"] = df_keep["exchange"].astype(str).str.strip().str.upper()
                counts = df_keep["exchange"].value_counts()
                for ex in ["HOSE", "HNX", "UPCOM"]:
                    logger.info("Listing %s: %s mã", ex, int(counts.get(ex, 0)))
            else:
                logger.info("Listing: không thấy cột exchange")

            logger.info(
                "Lấy Listing thành công (%s mã) từ source=%s",
                len(df_keep),
                src.upper(),
            )
            return df_keep

        except Exception as e:
            last_err = e
            logger.warning("Lỗi fetch_listing source=%s: %s", src, e)
            continue

    logger.warning("Không lấy được Listing: %s", last_err)
    return pd.DataFrame()


# --- Run Listing ---
_ensure_ingestion_workdir()
listing_df = fetch_listing()

try:
    base_dir = os.path.abspath(
        os.path.join(os.getcwd(), "..", "data-lake", "raw", "listing")
    )
    os.makedirs(base_dir, exist_ok=True)

    parquet_path = os.path.join(base_dir, "listing.parquet")
    csv_path = os.path.join(base_dir, "listing.csv")

    if listing_df.empty:
        logger.warning("Bỏ qua lưu Listing vì DataFrame rỗng.")
    else:
        try:
            listing_df.to_parquet(parquet_path, engine="pyarrow", index=False)
        except ImportError:
            listing_df.to_parquet(parquet_path, engine="fastparquet", index=False)
        listing_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        logger.info("Đã ghi %s dòng → %s", len(listing_df), parquet_path)
        logger.info("Đã xuất %s dòng → %s", len(listing_df), csv_path)

except Exception:
    logger.exception("Lỗi khi lưu Listing")


2026-04-03 17:17:58 [INFO] Listing: không thấy cột exchange
2026-04-03 17:17:58 [INFO] Lấy Listing thành công (1544 mã) từ source=KBS
2026-04-03 17:17:58 [INFO] Đã ghi 1544 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\listing\listing.parquet
2026-04-03 17:17:58 [INFO] Đã xuất 1544 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\listing\listing.csv


In [9]:
def fetch_company_overview(ticker: str) -> dict:
    """Lấy thông tin tổng quan doanh nghiệp cho một mã.

    Không raise exception; lỗi sẽ trả về dict {'ticker': ticker}.
    """
    sym = ticker.upper().strip()
    try:
        company = Company(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        result = None
        try:
            if hasattr(company, "overview"):
                result = company.overview()
            elif hasattr(company, "profile"):
                result = company.profile()
            else:
                logger.warning("Company: không có method overview/profile (ticker=%s)", sym)
                result = {}
        except Exception as e:
            logger.warning("Company method lỗi %s: %s", sym, e)
            result = {}

        # Chuẩn hóa về dict
        if result is None:
            out: dict = {}
        elif isinstance(result, dict):
            out = dict(result)
        elif isinstance(result, pd.Series):
            out = result.to_dict()
        elif isinstance(result, pd.DataFrame):
            out = result.iloc[0].to_dict() if not result.empty else {}
        else:
            try:
                out = dict(result)
            except Exception:
                out = {}

        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_company_overview lỗi %s: %s", sym, e)
        return {"ticker": sym}


# --- Run company overview ---
_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("Bắt đầu thu thập company overview (%s mã)", len(TICKERS))

rows: list[dict] = []
total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải company %s...", idx + 1, total, ticker)
    try:
        rows.append(fetch_company_overview(ticker))
    except Exception:
        logger.exception("Lỗi khi xử lý company %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(0.5)

company_df = pd.DataFrame(rows)

try:
    if company_df.empty:
        logger.warning("company_df rỗng — bỏ qua lưu.")
    else:
        save_raw(company_df, "company", run_date, "company_overview")
except Exception:
    logger.exception("Lỗi khi lưu company_overview")


2026-04-03 17:17:58 [INFO] Bắt đầu thu thập company overview (50 mã)
2026-04-03 17:17:58 [INFO] [1/50] Đang tải company ACB...
2026-04-03 17:17:59 [INFO] [2/50] Đang tải company BCM...
2026-04-03 17:18:00 [INFO] [3/50] Đang tải company BID...
2026-04-03 17:18:01 [INFO] [4/50] Đang tải company BVH...
2026-04-03 17:18:02 [INFO] [5/50] Đang tải company CTG...
2026-04-03 17:18:03 [INFO] [6/50] Đang tải company FPT...
2026-04-03 17:18:04 [INFO] [7/50] Đang tải company GAS...



⚠️ 
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 56 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 56 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia gói thành viên tài trợ để sử dụng không bị gián đoạn
      (Join sponsor membership for uninterrupted access)

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     Tham gia: https://vnstocks.com/insiders-program
     Sau khi tham gia tài trợ, cài bộ thư viện riêng vnstock_data theo hướng dẫn https://vnstocks.com/onboard-member





SystemExit: Rate limit exceeded. 
============================================================
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)
============================================================

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 56 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 56 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia gói thành viên tài trợ để sử dụng không bị gián đoạn
      (Join sponsor membership for uninterrupted access)

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     Tham gia: https://vnstocks.com/insiders-program
     Sau khi tham gia tài trợ, cài b

To exit: use 'exit', 'quit', or Ctrl-D.


In [ ]:
def fetch_financial_ratio(ticker: str) -> pd.DataFrame:
    """Lấy financial ratio theo quý (fallback year nếu lỗi)."""
    sym = ticker.upper().strip()
    try:
        finance = Finance(source=PRIMARY_QUOTE_SOURCE, symbol=sym)

        df = pd.DataFrame()
        try:
            df = finance.ratio(period="quarter")
        except Exception:
            df = finance.ratio(period="year")

        if df is None or (hasattr(df, "empty") and df.empty):
            return pd.DataFrame()

        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)

        out = df.copy()
        out["ticker"] = sym
        return out

    except Exception as e:
        logger.warning("fetch_financial_ratio lỗi %s: %s", sym, e)
        return pd.DataFrame()


# --- Run financial_ratio ---
_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("Bắt đầu thu thập financial_ratio (%s mã) theo quarter", len(TICKERS))

total = len(TICKERS)
for idx, ticker in enumerate(TICKERS):
    logger.info("[%s/%s] Đang tải financial ratio %s...", idx + 1, total, ticker)
    try:
        df_ratio = fetch_financial_ratio(ticker)
        if df_ratio.empty:
            logger.warning("Bỏ qua %s — financial_ratio rỗng", ticker)
        else:
            save_raw(df_ratio, "financial_ratio", run_date, ticker)
            logger.info("Thành công ratio %s (%s dòng)", ticker, len(df_ratio))
    except Exception:
        logger.exception("Lỗi khi xử lý financial_ratio %s — tiếp tục", ticker)

    if idx < total - 1:
        time.sleep(1)

logger.info("Hoàn tất financial_ratio cho %s mã.", total)


In [ ]:
def fetch_price_board(tickers_list: list[str]) -> pd.DataFrame:
    """Thu thập snapshot bảng giá tham chiếu/trần/sàn.

    Nếu API/method không khả dụng sẽ log warning và trả DataFrame rỗng.
    """
    from datetime import datetime, time as dtime
    import re

    required_norm_map = {
        "ticker": "ticker",
        "ref_price": "ref_price",
        "ceiling_price": "ceiling_price",
        "floor_price": "floor_price",
        "match_price": "match_price",
        "match_volume": "match_volume",
        "foreign_buy_vol": "foreign_buy_vol",
        "foreign_sell_vol": "foreign_sell_vol",
        "bid_price1": "bid_price1",
        "bid_vol1": "bid_vol1",
        "ask_price1": "ask_price1",
        "ask_vol1": "ask_vol1",
    }

    def _norm_col(s: str) -> str:
        return re.sub(r"[\s_]+", "", str(s).strip().lower())

    # Chỉ chạy trong giờ giao dịch theo yêu cầu.
    now_t = datetime.now().time()
    if not (dtime(9, 0) <= now_t <= dtime(15, 0)):
        logger.info("Ngoài giờ giao dịch (now=%s) — bỏ qua price_board.", now_t.strftime("%H:%M:%S"))
        return pd.DataFrame()

    if not tickers_list:
        return pd.DataFrame()

    try:
        probe = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=tickers_list[0])
        if not hasattr(probe, "price_board"):
            logger.warning("Method `Quote.price_board()` không khả dụng trong vnstock version này.")
            return pd.DataFrame()

        df_out = pd.DataFrame()

        # 1) Thử lấy nhiều mã cùng lúc (nếu method hỗ trợ tham số symbols/tickers).
        try:
            try:
                df_out = probe.price_board(symbols=tickers_list)
            except TypeError:
                df_out = probe.price_board(tickers_list)
        except Exception:
            df_out = pd.DataFrame()

        # 2) Fallback: lấy lần lượt từng mã nếu không có multi.
        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            frames: list[pd.DataFrame] = []
            for t in tickers_list:
                try:
                    q = Quote(source=PRIMARY_QUOTE_SOURCE, symbol=t)
                    sub = q.price_board()
                    if sub is None or (hasattr(sub, "empty") and sub.empty):
                        continue
                    if not isinstance(sub, pd.DataFrame):
                        sub = pd.DataFrame(sub)
                    sub = sub.copy()
                    if "ticker" not in sub.columns:
                        sub["ticker"] = t
                    frames.append(sub)
                    # Không sleep để tránh quá chậm; nếu rate limit bạn có thể thêm sleep.
                except Exception as e:
                    logger.warning("price_board lỗi %s: %s", t, e)

            if frames:
                df_out = pd.concat(frames, ignore_index=True)

        if df_out is None or (hasattr(df_out, "empty") and df_out.empty):
            return pd.DataFrame()

        if not isinstance(df_out, pd.DataFrame):
            df_out = pd.DataFrame(df_out)

        # Chỉ giữ các cột cần nếu có.
        col_map = {_norm_col(c): c for c in df_out.columns}
        required_norms = [_norm_col(k) for k in required_norm_map.keys()]
        keep_cols = [col_map[rn] for rn in required_norms if rn in col_map]
        out = df_out.loc[:, keep_cols].copy() if keep_cols else df_out.copy()

        # Đảm bảo cột ticker luôn có.
        if "ticker" not in [_norm_col(c) for c in out.columns] and "ticker" not in out.columns:
            out["ticker"] = tickers_list[0]

        return out

    except Exception as e:
        logger.warning("fetch_price_board lỗi: %s", e)
        return pd.DataFrame()


# --- Run price_board snapshot ---
_ensure_ingestion_workdir()
run_dt = date.today()
run_date = run_dt.isoformat()

logger.info("Bắt đầu thu thập price_board snapshot (%s mã)", len(TICKERS))

try:
    board_df = fetch_price_board(TICKERS)
    if board_df.empty:
        logger.warning("price_board_snapshot rỗng — bỏ qua lưu.")
    else:
        save_raw(board_df, "price_board", run_date, "price_board_snapshot")
except Exception:
    logger.exception("Lỗi khi thu thập/lưu price_board")


In [ ]:
# CELL 9 — Kiểm tra tổng kết dữ liệu theo từng category.

import logging
from datetime import date
from pathlib import Path

import pandas as pd

logger = globals().get("logger", logging.getLogger(__name__))


def _datalake_raw_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_price_data.ipynb").is_file():
        base = cwd
    elif (cwd / "ingestion" / "ingest_price_data.ipynb").is_file():
        base = cwd / "ingestion"
    else:
        base = cwd
    return (base / ".." / "data-lake" / "raw").resolve()


raw_root = _datalake_raw_root()
run_date = date.today().isoformat()

categories = [
    "price",
    "index",
    "foreign_trading",
    "listing",
    "company",
    "financial_ratio",
    "price_board",
]


def _read_parquet_rows(p: Path) -> int:
    try:
        df = pd.read_parquet(p)
        return int(len(df))
    except Exception as e:
        logger.warning("QC: đọc lỗi %s: %s", p, e)
        return 0


summary_rows: list[dict] = []
missing_or_empty: list[str] = []

for cat in categories:
    total_rows = 0
    file_count = 0

    if cat == "listing":
        p = raw_root / "listing" / "listing.parquet"
        if p.is_file():
            file_count = 1
            total_rows = _read_parquet_rows(p)
    else:
        folder = raw_root / cat / f"date={run_date}"
        if folder.is_dir():
            parquets = sorted(folder.glob("*.parquet"))
            file_count = len(parquets)
            total_rows = sum(_read_parquet_rows(x) for x in parquets)

    if file_count == 0:
        status = "MISSING"
    elif total_rows == 0:
        status = "EMPTY"
    else:
        status = "OK"

    if status != "OK":
        missing_or_empty.append(cat)

    summary_rows.append(
        {
            "Category": cat,
            "Files": file_count,
            "Total rows": total_rows,
            "Status": status,
        }
    )


print("Category           | Files | Total rows | Status")
print("------------------|-------|------------|--------")
for r in summary_rows:
    print(
        f"{r['Category']:<18} | {r['Files']:<5} | {r['Total rows']:<10,} | {r['Status']}"
    )

if missing_or_empty:
    print("\n[QC WARNING] Các category bị thiếu hoặc rỗng:")
    for cat in missing_or_empty:
        print(f"- {cat}")
else:
    print("\n[QC] Tất cả category có dữ liệu.")

